# Pre-processing of ESCO data
Felix Zaussinger | 14.02.2022

## Core Analysis Goal(s)
1. Calculate occupation-skills and occupation similarity matrices based on ESCO v.1.1.0
    - occupation-skills matrix: unweighted, essential/optional
    - occupation-similarity matrix: co-occurrence (optional: cosine similarity, relatedness, etc.)
2. Configure static parameters in config file
3. Create skills metadata file (green/non-green, coreness)

## Key Insight(s)
1.
2.
3.

In [2]:
import os
import sys
import logging
from pathlib import Path

import yaml
import numpy as np
from tqdm import tqdm

%load_ext autoreload
%autoreload 2

import pandas as pd
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 120)

logging.basicConfig(level=logging.INFO, stream=sys.stdout)

Define directory structure

In [3]:
# project directory
abspath = os.path.abspath('')
project_dir = str(Path(abspath).parents[0])

# sub-directories
data_raw = os.path.join(project_dir, "data", "raw")
data_interim = os.path.join(project_dir, "data", "interim")
data_processed = os.path.join(project_dir, "data", "processed")
figure_dir = os.path.join(project_dir, "reports", "figures")

Import config file

In [4]:
# folder to load config file
CONFIG_FOLDER = "configs"

# Function to load yaml configuration file
def load_config(config_name):
    with open(os.path.join(project_dir, CONFIG_FOLDER, config_name)) as file:
        config_file = yaml.safe_load(file)

    return config_file

# load
config = load_config("main_config.yml")
config

{'ESCO': {'LANGUAGE': 'en',
  'VERSION': 'v1.1.0',
  'WEIGHT_UNIFORM': 1,
  'WEIGHT_ESSENTIAL_SKILL': 1,
  'WEIGHT_OPTIONAL_SKILL': 0.5},
 'TRANSITION_ANALYSIS': {'MIN_VIABLE': 0.3,
  'HIGHLY_VIABLE': 0.4,
  'MAX_JOB_ZONE_DIF': 1,
  'MIN_EARNINGS_RATIO': 0.75}}

#### Load ESCO data

In [5]:
# esco configurations
esco_language = config["ESCO"]["LANGUAGE"] # "en"
esco_version = config["ESCO"]["VERSION"] # "v1.0.3" or "v1.1.0"
esco_skills_hierarchy_version = "v1.0.8" if esco_version == "v1.0.3" else "v1.1.0"

# core
occ = pd.read_csv(os.path.join(data_raw, "esco", esco_version, "occupations_{}.csv".format(esco_language)))
skills = pd.read_csv(os.path.join(data_raw, "esco", esco_version, "skills_{}.csv".format(esco_language)))
occ_skills_mapping = pd.read_csv(os.path.join(data_raw, "esco", esco_version, "occupationSkillRelations.csv"))

# additional
skill_groups = pd.read_csv(os.path.join(data_raw, "esco", esco_version, "skillGroups_{}.csv".format(esco_language)))
skills_hierarchy = pd.read_csv(os.path.join(data_raw, "esco", esco_skills_hierarchy_version, "skillsHierarchy_{}.csv".format(esco_language)))

skills_hierarchy_kanders = pd.read_csv(os.path.join(data_raw, "mapping-career-causeways", "codebase", "data", "processed", "ESCO_skills_hierarchy", "ESCO_skills_hierarchy.csv"))

### Build occupation-skills matrix

**Encoding**
- 0: skill not required
- 1: skill required, essential
- 2: skill required, optional

In [6]:
%%time

target_path = os.path.join(project_dir, "data", "interim", "esco", esco_version, "occ_skills_matrix_weighted.pkl")

if not os.path.exists(target_path):
    errors = 0
    skill_vectors = []

    for i in tqdm(range(len(occ))):
        occ_uri = occ.iloc[i, :][1]

        # lookup corresponding skills
        skill_list = occ_skills_mapping[occ_skills_mapping["occupationUri"] == occ_uri]

        # create vector
        skill_vector = []
        for j, skill in enumerate(skills.conceptUri.values):

            if skill in skill_list.skillUri.values:
                relation_type = skill_list.loc[skill_list.skillUri == skill, "relationType"].values[0]

                # skill needed for occupation and essential
                if relation_type == "essential":
                    skill_vector.append(1)
                # skill needed for occupation and optional
                elif relation_type == "optional":
                    skill_vector.append(2)
            else:
                # skill not needed for occupation
                skill_vector.append(0)

        indices = [i for i, j in enumerate(skill_vector) if j == 1]

        # sanity check
        if len(skill_list.skillUri) != np.sum(np.invert(np.array(skill_vector) == 0)):
            errors += 1

        # append
        skill_vectors.append(skill_vector)

    # info
    print("n_errors: ", errors)

    # create df
    occ_skills_matrix_eo = pd.DataFrame(
        index=occ.conceptUri,
        columns=skills.conceptUri,
        data=np.array(skill_vectors)
    )

    # save to disk
    occ_skills_matrix_eo.to_pickle(target_path)
else:
    # read from disk
    occ_skills_matrix_eo = pd.read_pickle(target_path)

CPU times: user 3.33 ms, sys: 413 ms, total: 417 ms
Wall time: 499 ms


Calculate weighted and unweighted variants

In [7]:
# weighted form
replace_weighted = {
    1: config["ESCO"]["WEIGHT_ESSENTIAL_SKILL"],
    2: config["ESCO"]["WEIGHT_OPTIONAL_SKILL"]
}
occ_skills_matrix_weighted = occ_skills_matrix_eo.replace(to_replace=replace_weighted)

# unweighted form
occ_skills_matrix_unweighted = occ_skills_matrix_eo.replace(to_replace=[1, 2], value=config["ESCO"]["WEIGHT_UNIFORM"])

#### Calculate (co-occurrence) occupation similarity matrix

Weighted form

In [8]:
%%time

target_path = os.path.join(project_dir, "data", "interim", "esco", esco_version, "occ_sim_matrix_weighted_coo.pkl")

if not os.path.exists(target_path):

    # calculate co-occurrence matrix via matrix multiplication with transpose form
    occ_sim_matrix_weighted_coo = np.dot(
        occ_skills_matrix_weighted.values,
        occ_skills_matrix_weighted.values.transpose()
    )

    # to df
    df_occ_sim_matrix_weighted_coo = pd.DataFrame(
        index=occ.conceptUri,
        columns=occ.conceptUri,
        data=occ_sim_matrix_weighted_coo
    )

    # save
    df_occ_sim_matrix_weighted_coo.to_pickle(target_path)
else:
    # read
    df_occ_sim_matrix_weighted_coo = pd.read_pickle(target_path)

CPU times: user 1.26 ms, sys: 43.7 ms, total: 44.9 ms
Wall time: 91.5 ms


Unweighted form

%%time

target_path = os.path.join(project_dir, "data", "interim", "esco", esco_version, "occ_sim_matrix_unweighted_coo.pkl")

if not os.path.exists(target_path):

    # calculate co-occurrence matrix via matrix multiplication with transpose form
    occ_sim_matrix_unweighted_coo = np.dot(
        occ_skills_matrix_unweighted.values,
        occ_skills_matrix_unweighted.values.transpose()
    )

    # to df
    df_occ_sim_matrix_unweighted_coo = pd.DataFrame(
        index=occ.conceptUri,
        columns=occ.conceptUri,
        data=occ_sim_matrix_unweighted_coo
    )

    # save
    df_occ_sim_matrix_unweighted_coo.to_pickle(target_path)
else:
    # read
    df_occ_sim_matrix_unweighted_coo = pd.read_pickle(target_path)

#### Create ESCO Skills Metadata File

Green Skills

In [65]:
# read green skill data
green_skills = pd.read_csv(os.path.join(data_raw, "esco", esco_version, "greenSkillsCollection_{}.csv".format(esco_language)))
green_id_colname = "skillGreen"
green_skills[green_id_colname] = True

In [66]:
# find cols that are unique in green skills file compared to general skills file
# https://www.kaggle.com/ashukr/sets-and-venn-diagram-in-python: The difference between A and B contains all elements that are in A but not in B
set_diff = list(set(green_skills.columns.values.tolist()) - set(skills.columns.values.tolist()))
set_diff.insert(0, "conceptUri")

In [78]:
# copy skills df and join information on green skills
skills_metadata = skills.copy()
skills_metadata = skills_metadata.merge(right=green_skills[set_diff], on="conceptUri", how="left", validate="one_to_one")
skills_metadata = skills_metadata.fillna(value={green_id_colname: False})

Coreness

In [79]:
skills_coreness = pd.read_csv(
    os.path.join(data_raw, "mapping-career-causeways", "codebase", "data", "interim", "upskilling_analysis", "skills_coreness_measure.csv")
)

# TODO: merge via conceptUri of ESCO v1.0.3 skills
keep_cols = ["preferred_label", "coreness"]
skills_metadata = skills_metadata.merge(right=skills_coreness[keep_cols], left_on="preferredLabel", right_on="preferred_label", how="left", validate="one_to_one")

Evaluation

In [83]:
print(len(skills))  # new esco
print(len(skills_coreness))  # old esco
print(len(skills_metadata.coreness.dropna()))  # after merge

print(len(skills_coreness) - len(skills_metadata.coreness.dropna()))  # (probably) existing in v1.1.0 but unmatched

13891
13485
12988
497
